# DSAI 413 — Assignment 2 · Kaggle runbook

ColPali-RAG vs. MedGemma on MIMIC-CXR. Free T4 16 GB.

Before you click Run All, in the notebook sidebar:
1. **Add data** → `simhadrisadaram/mimic-cxr-dataset`.
2. **Settings → Accelerator** → `GPU T4 x1`.
3. **Settings → Internet** → On.
4. **Add-ons → Secrets** → add `HF_TOKEN` and `ANTHROPIC_API_KEY`.
5. **Edit `REPO_URL` below** to point at your fork.

In [ ]:
import os, subprocess
os.chdir('/kaggle/working')

# If repo is public, plain clone works. If private, add a GH_TOKEN secret
# in Kaggle (Add-ons -> Secrets) and we'll embed it in the URL for auth.
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    gh_token = UserSecretsClient().get_secret('GH_TOKEN')
    if gh_token and url.startswith('https://github.com/'):
        url = url.replace('https://github.com/', f'https://{gh_token}@github.com/')
        print('Using GH_TOKEN for private repo auth.')
except Exception:
    pass  # no GH_TOKEN secret -> assume public repo

if not os.path.exists('cxr-rag'):
    subprocess.check_call(['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, url, 'cxr-rag'])
os.chdir('/kaggle/working/cxr-rag')
print('cwd:', os.getcwd())
!ls

## 1. Secrets, repo, and dependencies

In [ ]:
import os, subprocess, sys
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ['HF_TOKEN']          = secrets.get_secret('HF_TOKEN')
os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']
try:
    os.environ['ANTHROPIC_API_KEY'] = secrets.get_secret('ANTHROPIC_API_KEY')
except Exception:
    print('No ANTHROPIC_API_KEY secret — VQA generation step will fail unless you set OPENAI_API_KEY instead.')
print('Tokens loaded.')

In [ ]:
import os, subprocess
os.chdir('/kaggle/working')
if not os.path.exists('cxr-rag'):
    subprocess.check_call(['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, REPO_URL, 'cxr-rag'])
os.chdir('/kaggle/working/cxr-rag')
print('cwd:', os.getcwd())
!ls

In [ ]:
# Kaggle's base image already has torch + transformers + Pillow + pandas.
# We only need the extra retrieval / metric libraries.
!pip install -q --no-deps colpali-engine==0.3.4 peft==0.13.0 einops==0.8.0
!pip install -q rouge-score==0.1.2 bert-score==0.3.13 nltk==3.9.1 sacrebleu==2.4.3
!pip install -q anthropic==0.39.0
!pip install -q bitsandbytes==0.43.3
print('deps installed')

In [ ]:
import torch, transformers, sys
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('transformers', transformers.__version__)
print('python', sys.version.split()[0])

## 2. Data preparation

The Kaggle dataset is mounted read-only at `/kaggle/input/mimic-cxr-dataset`. We point the preprocess step at it directly; no Kaggle CLI download needed.

In [ ]:
CFG = 'configs/kaggle.yaml'
!ls /kaggle/input/mimic-cxr-dataset | head -20

In [ ]:
# preprocess.py auto-discovers the reports CSV and image dir under data_root
# (configs/kaggle.yaml sets data_root to /kaggle/input/mimic-cxr-dataset).
!python -m data_prep.preprocess_kaggle --config $CFG

In [ ]:
!python -m data_prep.split --config $CFG

## 3. Build the VQA dataset (Claude Haiku as the LLM judge)

Skip this cell if you only care about Mode A (report generation).

In [ ]:
!python -m data_prep.build_vqa_dataset --config $CFG
!python -m data_prep.split_vqa --config $CFG

## 4. Build the ColPali index over the training corpus

In [ ]:
!python -m retrieval.colpali_index --config $CFG

## 5. Generate predictions — Mode A (report) and Mode B (VQA)

Each mode runs both systems (ColPali-RAG and MedGemma-only baseline).

In [ ]:
!python -m rag.run_mode_a --config $CFG --system both

In [ ]:
!python -m rag.run_mode_b --config $CFG --system both

## 6. Evaluate

In [ ]:
!python -m eval.metrics_report  --config $CFG
!python -m eval.metrics_vqa     --config $CFG
!python -m eval.qualitative_dump --config $CFG

In [ ]:
import json, pandas as pd
a = json.load(open('/kaggle/working/outputs/metrics_mode_a.json'))
b = json.load(open('/kaggle/working/outputs/metrics_mode_b.json'))
print('=== Mode A ===')
display(pd.DataFrame({k: v for k, v in a.items() if k in ('rag','baseline')}).T)
print('=== Mode B ===')
display(pd.DataFrame({k: v for k, v in b.items() if k in ('rag','baseline')}).T)

## 7. Persist outputs

Kaggle keeps everything under `/kaggle/working` after the session ends, downloadable from the notebook's "Output" tab. Copy these files back into the repo's `outputs/` directory for the report and demo:
- `metrics_mode_a.json` and `metrics_mode_b.json`
- `mode_a_{rag,baseline}.jsonl` and `mode_b_{rag,baseline}.jsonl`
- `qualitative_mode_a.md` and `qualitative_mode_b.md`